# AI Image Editor — CV showcase (all 9 modes)

This notebook runs every public editing mode with deterministic settings. Each mode:

- preprocesses every input image before inference;
- prints processed resolutions, runtime, backend, seed/steps, and per-stage peak VRAM;
- displays the processed input, masks/reference where applicable, and final output;
- saves README-ready PNG files plus per-mode JSON metrics under `outputs/showcase/`;
- incrementally updates `outputs/showcase/run-summary.json` after every successful mode.

> Peak VRAM is measured by the project's `MemoryTracker`. With `COLD_START_EACH_MODE=True`, models are released before each mode so measurements are easier to compare.

In [ ]:
from pathlib import Path
import gc
import json
import time
from datetime import datetime, timezone

import numpy as np
from PIL import Image, ImageDraw
from IPython.display import Markdown, display

from inpaint_core import (
    BoxPrompt,
    GenerationOptions,
    ImageProcessor,
    OutpaintMargins,
    PointPrompt,
    UpscaleOptions,
    preprocess_image,
    preprocess_image_only,
)

current = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (current, *current.parents) if (path / 'pyproject.toml').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not find the project root containing pyproject.toml')

ASSETS = PROJECT_ROOT / 'assets'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'showcase'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

processor = ImageProcessor.from_config(PROJECT_ROOT / 'configs' / 'default.yaml')
PROCESSING_MAX_SIDE = int(processor.config.processing.max_image_side)
OMNIPAINT_MAX_SIDE = int(processor.config.omnipaint.options['max_image_side'])
REFERENCE_MAX_SIDE = int(processor.config.omnipaint.options.get('reference_size', 512))
COLD_START_EACH_MODE = True
RUN_SUMMARY = {}

print('Project root:', PROJECT_ROOT)
print('Showcase output:', OUTPUT_DIR)
print('Default processed image max side:', PROCESSING_MAX_SIDE)
print('OmniPaint target max side:', OMNIPAINT_MAX_SIDE)
print('Processed reference max side:', REFERENCE_MAX_SIDE)
print('Cold start per mode:', COLD_START_EACH_MODE)

In [ ]:
def resolution(array: np.ndarray) -> str:
    return f'{array.shape[1]}x{array.shape[0]}'


def prepare_image(filename: str, *, max_side: int = PROCESSING_MAX_SIDE) -> np.ndarray:
    return preprocess_image_only(ASSETS / filename, max_side=max_side)


def prepare_image_with_point(
    filename: str,
    x_fraction: float | None = None,
    y_fraction: float | None = None,
    *,
    x_pixel: float | None = None,
    y_pixel: float | None = None,
    max_side: int = PROCESSING_MAX_SIDE,
):
    """Prepare an image and scale one positive point with it.

    Fractions are the primary interface and refer to the original image:
        prepare_image_with_point('image.jpg', 0.5, 0.5)

    Exact original-image pixels are also accepted as keyword arguments:
        prepare_image_with_point('image.jpg', x_pixel=640, y_pixel=420)
    """
    raw = preprocess_image_only(ASSETS / filename)
    fraction_supplied = x_fraction is not None or y_fraction is not None
    pixel_supplied = x_pixel is not None or y_pixel is not None

    if fraction_supplied and pixel_supplied:
        raise ValueError('Use either fractions or pixels, not both.')
    if pixel_supplied:
        if x_pixel is None or y_pixel is None:
            raise ValueError('x_pixel and y_pixel must be provided together.')
        x, y = float(x_pixel), float(y_pixel)
    else:
        if x_fraction is None or y_fraction is None:
            raise ValueError('Provide x_fraction/y_fraction or x_pixel/y_pixel.')
        if not 0 <= x_fraction <= 1 or not 0 <= y_fraction <= 1:
            raise ValueError('Point fractions must be in the [0, 1] range.')
        x = raw.shape[1] * x_fraction
        y = raw.shape[0] * y_fraction

    if not 0 <= x <= raw.shape[1] or not 0 <= y <= raw.shape[0]:
        raise ValueError(f'Point ({x}, {y}) is outside the original {resolution(raw)} image.')
    selection = PointPrompt(points=[(x, y)], labels=[1])
    return preprocess_image(raw, selection, max_side=max_side)


def selection_overlay(
    image: np.ndarray,
    selection,
    *,
    color=(255, 40, 90),
    fill_opacity: int = 72,
) -> np.ndarray:
    """Return a display copy with a visible point or translucent box overlay."""
    base = Image.fromarray(image).convert('RGBA')
    layer = Image.new('RGBA', base.size, (0, 0, 0, 0))
    draw = ImageDraw.Draw(layer)
    width, height = base.size
    line_width = max(2, round(min(width, height) * 0.006))

    if isinstance(selection, PointPrompt):
        radius = max(5, round(min(width, height) * 0.012))
        for x, y in selection.points:
            bounds = (x - radius, y - radius, x + radius, y + radius)
            draw.ellipse(bounds, fill=(*color, 235), outline=(255, 255, 255, 255), width=line_width)
    elif isinstance(selection, BoxPrompt):
        bounds = (selection.x1, selection.y1, selection.x2, selection.y2)
        draw.rectangle(bounds, fill=(*color, fill_opacity), outline=(*color, 255), width=line_width)
    else:
        raise TypeError('selection must be PointPrompt or BoxPrompt')

    return np.ascontiguousarray(np.asarray(Image.alpha_composite(base, layer).convert('RGB')))


def save_and_display(mode_id: str, items):
    for slug, title, array in items:
        path = OUTPUT_DIR / f'{mode_id}-{slug}.png'
        Image.fromarray(array).save(path)
        display(Markdown(f'**{title}** — `{resolution(array)}`  '))
        display(Image.fromarray(array))
        print('Saved:', path)


def profile_mode(mode_id: str, label: str, workflow):
    if COLD_START_EACH_MODE:
        processor.release_models()
        gc.collect()
        try:
            import torch
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except ImportError:
            pass

    processor.memory_tracker.results.clear()
    started = time.perf_counter()
    payload = workflow()
    elapsed_s = time.perf_counter() - started
    stages = processor.memory_tracker.to_dict()

    allocated = [v['peak_allocated_mb'] for v in stages.values() if v['peak_allocated_mb'] is not None]
    reserved = [v['peak_reserved_mb'] for v in stages.values() if v['peak_reserved_mb'] is not None]
    peak_allocated = max(allocated) if allocated else None
    peak_reserved = max(reserved) if reserved else None

    result = payload['result']
    metadata = result.metadata
    backend_name = metadata.get('backend')
    backend_configs = {
        'flux2_klein': processor.config.flux2_klein,
        'flux_fill': processor.config.flux_fill,
        'omnipaint': processor.config.omnipaint,
        'realesrgan': processor.config.upscaler,
    }
    backend_config = backend_configs.get(backend_name)
    resolutions = {
        key: resolution(value)
        for key, value in payload.items()
        if isinstance(value, np.ndarray) and not key.endswith('_overlay')
    }
    resolutions['output'] = resolution(result.image)

    gpu_name = None
    try:
        import torch
        if torch.cuda.is_available():
            gpu_name = torch.cuda.get_device_name(torch.cuda.current_device())
    except ImportError:
        pass

    record = {
        'mode_id': mode_id,
        'label': label,
        'generated_at': datetime.now(timezone.utc).isoformat(),
        'cold_start': COLD_START_EACH_MODE,
        'gpu': gpu_name,
        'backend': backend_name,
        'model_id': metadata.get('model_id') or (backend_config.model_id if backend_config else None),
        'seed': metadata.get('seed', getattr(result, 'seed', None)),
        'num_inference_steps': metadata.get('num_inference_steps'),
        'runtime_seconds': round(elapsed_s, 4),
        'peak_vram_allocated_mb': peak_allocated,
        'peak_vram_reserved_mb': peak_reserved,
        'resolutions': resolutions,
        'parameters': payload.get('parameters', {}),
        'stages': stages,
    }
    metrics_path = OUTPUT_DIR / f'{mode_id}-run.json'
    RUN_SUMMARY[mode_id] = record
    metrics_path.write_text(json.dumps(record, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')
    summary_path = OUTPUT_DIR / 'run-summary.json'
    summary_path.write_text(
        json.dumps({'modes': RUN_SUMMARY}, indent=2, ensure_ascii=False) + '\n',
        encoding='utf-8',
    )
    print(f'\n=== {label} ===')
    print(f'Total runtime: {elapsed_s:.2f} s')
    print('Backend:', metadata.get('backend', 'n/a'))
    print('Seed:', metadata.get('seed', getattr(result, 'seed', 'n/a')))
    print('Inference steps:', metadata.get('num_inference_steps', 'n/a'))
    print('Resolutions:', resolutions)
    print('Peak VRAM allocated:', f'{peak_allocated:.1f} MB' if peak_allocated is not None else 'n/a (CUDA unavailable)')
    print('Peak VRAM reserved:', f'{peak_reserved:.1f} MB' if peak_reserved is not None else 'n/a (CUDA unavailable)')
    print('Per-stage measurements:')
    for stage, stats in stages.items():
        alloc = stats['peak_allocated_mb']
        reserve = stats['peak_reserved_mb']
        print(
            f"  {stage}: {stats['elapsed_ms'] / 1000:.2f} s | "
            f"allocated={alloc:.1f} MB | reserved={reserve:.1f} MB"
            if alloc is not None and reserve is not None
            else f"  {stage}: {stats['elapsed_ms'] / 1000:.2f} s | VRAM=n/a"
        )
    print('Saved run parameters:', metrics_path)
    print('Updated summary:', summary_path)
    return payload


GEN_KLEIN = GenerationOptions(seed=42, num_inference_steps=4, guidance_scale=1.0)
GEN_FLUX_FILL = GenerationOptions(seed=42, num_inference_steps=28, guidance_scale=30.0)
GEN_OMNIPAINT = GenerationOptions(seed=42, num_inference_steps=28, guidance_scale=3.5)

## 1. Remove Object — OmniPaint
Remove the phone from the coffee table.

In [ ]:
def remove_workflow():
    image, point = prepare_image_with_point(
        'demo-coffee-table.jpg', 0.39, 0.84, max_side=OMNIPAINT_MAX_SIDE
    )
    segmentation = processor.segment(image, point)
    mask = segmentation.best_mask
    result = processor.remove_object(image, mask=mask, generation_options=GEN_OMNIPAINT)
    return {
        'result': result, 'input': image,
        'input_overlay': selection_overlay(image, point), 'mask': result.mask,
        'parameters': {
            'input_file': 'demo-coffee-table.jpg',
            'preprocess_max_side': OMNIPAINT_MAX_SIDE,
            'point_fraction': [0.39, 0.84],
            'point_processed_pixels': list(point.points[0]),
            'mask_options': dict(processor.config.mask_defaults),
        },
    }

remove_case = profile_mode('01-remove-object', '1. Remove Object', remove_workflow)
save_and_display('01-remove-object', [
    ('input', 'Processed input + point overlay', remove_case['input_overlay']),
    ('mask', 'Object mask', remove_case['mask']),
    ('output', 'Output', remove_case['result'].image),
])

## 2. Replace Object — FLUX Fill
Replace the front coffee cup with a small ceramic vase.

In [ ]:
def replace_workflow():
    image, point = prepare_image_with_point('demo-coffee-table.jpg', 0.66, 0.68)
    segmentation = processor.segment(image, point)
    mask = segmentation.best_mask
    prompt = 'a small white ceramic flower vase matching the perspective and warm cafe lighting'
    result = processor.replace_object(
        image,
        prompt,
        mask=mask,
        generation_options=GEN_FLUX_FILL,
    )
    return {
        'result': result, 'input': image,
        'input_overlay': selection_overlay(image, point), 'mask': result.mask,
        'parameters': {
            'input_file': 'demo-coffee-table.jpg',
            'preprocess_max_side': PROCESSING_MAX_SIDE,
            'point_fraction': [0.66, 0.68],
            'point_processed_pixels': list(point.points[0]),
            'prompt': prompt,
            'mask_options': dict(processor.config.mask_defaults),
        },
    }

replace_case = profile_mode('02-replace-object', '2. Replace Object', replace_workflow)
save_and_display('02-replace-object', [
    ('input', 'Processed input + point overlay', replace_case['input_overlay']),
    ('mask', 'Object mask', replace_case['mask']),
    ('output', 'Output', replace_case['result'].image),
])

## 3. Replace Background — FLUX.2 Klein
Keep the portrait subject and replace the greenery with a modern office.

In [ ]:
def background_workflow():
    image = prepare_image('demo-portrait.jpg')
    prompt = 'a bright modern creative office with soft window light and shallow depth of field'
    result = processor.replace_background(
        image,
        prompt,
        generation_options=GEN_KLEIN,
    )
    return {
        'result': result, 'input': image,
        'parameters': {
            'input_file': 'demo-portrait.jpg',
            'preprocess_max_side': PROCESSING_MAX_SIDE,
            'prompt': prompt,
        },
    }

background_case = profile_mode('03-replace-background', '3. Replace Background', background_workflow)
save_and_display('03-replace-background', [
    ('input', 'Processed input', background_case['input']),
    ('output', 'Output', background_case['result'].image),
])

## 4. Add Object by Prompt — FLUX.2 Klein
This example intentionally uses `placement=None`; the prompt determines location.

In [ ]:
def add_prompt_workflow():
    image = prepare_image('demo-mountain-field.jpg')
    prompt = 'a single colorful hot-air balloon floating naturally in the open sky above the field'
    result = processor.add_object_by_prompt(
        image,
        prompt,
        placement=None,
        generation_options=GEN_KLEIN,
    )
    return {
        'result': result, 'input': image,
        'parameters': {
            'input_file': 'demo-mountain-field.jpg',
            'preprocess_max_side': PROCESSING_MAX_SIDE,
            'prompt': prompt,
            'placement': None,
        },
    }

add_prompt_case = profile_mode('04-add-object-prompt', '4. Add Object by Prompt', add_prompt_workflow)
save_and_display('04-add-object-prompt', [
    ('input', 'Processed input', add_prompt_case['input']),
    ('output', 'Output', add_prompt_case['result'].image),
])

## 5. Add Object by Reference — OmniPaint
Select the chair from the reference and place it in the open ground area of the park.

In [ ]:
def add_reference_workflow():
    target = prepare_image('demo-park-bench.jpg', max_side=OMNIPAINT_MAX_SIDE)
    reference, reference_point = prepare_image_with_point(
        'demo-reference-chair.jpg', 0.76, 0.55, max_side=REFERENCE_MAX_SIDE
    )
    reference_segmentation = processor.segment(reference, reference_point)
    reference_mask = reference_segmentation.best_mask

    height, width = target.shape[:2]
    placement = BoxPrompt(
        x1=width * 0.72, y1=height * 0.38,
        x2=width * 0.94, y2=height * 0.78,
    )
    result = processor.add_object_by_reference(
        target,
        reference,
        placement=placement,
        reference_mask=reference_mask,
        generation_options=GEN_OMNIPAINT,
    )
    return {
        'result': result, 'input': target, 'reference': reference,
        'input_overlay': selection_overlay(target, placement),
        'reference_overlay': selection_overlay(reference, reference_point),
        'reference_mask': reference_mask, 'placement_mask': result.mask,
        'parameters': {
            'input_file': 'demo-park-bench.jpg',
            'reference_file': 'demo-reference-chair.jpg',
            'target_preprocess_max_side': OMNIPAINT_MAX_SIDE,
            'reference_preprocess_max_side': REFERENCE_MAX_SIDE,
            'reference_point_fraction': [0.76, 0.55],
            'reference_point_processed_pixels': list(reference_point.points[0]),
            'placement_fraction_xyxy': [0.72, 0.38, 0.94, 0.78],
            'placement_processed_pixels_xyxy': [
                placement.x1, placement.y1, placement.x2, placement.y2
            ],
            'mask_options': dict(processor.config.mask_defaults),
        },
    }

add_reference_case = profile_mode('05-add-object-reference', '5. Add Object by Reference', add_reference_workflow)
save_and_display('05-add-object-reference', [
    ('input', 'Processed target input + placement box', add_reference_case['input_overlay']),
    ('reference', 'Processed reference input + point overlay', add_reference_case['reference_overlay']),
    ('reference-mask', 'Reference subject mask', add_reference_case['reference_mask']),
    ('placement-mask', 'Target placement mask', add_reference_case['placement_mask']),
    ('output', 'Output', add_reference_case['result'].image),
])

## 6. Prompt Edit — FLUX.2 Klein
Restore the archival bus image as a modern natural-color photograph while preserving composition.

In [ ]:
def prompt_edit_workflow():
    image = prepare_image('demo-bus.jpg')
    prompt = 'Restore this archival photograph with natural modern colors, neutral white balance, and realistic contrast while preserving every person, object, and the original composition'
    result = processor.prompt_edit(
        image,
        prompt,
        generation_options=GEN_KLEIN,
    )
    return {
        'result': result, 'input': image,
        'parameters': {
            'input_file': 'demo-bus.jpg',
            'preprocess_max_side': PROCESSING_MAX_SIDE,
            'prompt': prompt,
        },
    }

prompt_edit_case = profile_mode('06-prompt-edit', '6. Prompt Edit', prompt_edit_workflow)
save_and_display('06-prompt-edit', [
    ('input', 'Processed input', prompt_edit_case['input']),
    ('output', 'Output', prompt_edit_case['result'].image),
])

## 7. Text-to-Image — FLUX.2 Klein
Generate a standalone portfolio image. There is no source image for this mode.

In [ ]:
def generation_workflow():
    prompt = 'A cinematic alpine lake at sunrise, mirror-like water, mist between mountains, realistic landscape photography, detailed natural lighting'
    result = processor.generate_image(
        prompt,
        width=1024,
        height=1024,
        generation_options=GEN_KLEIN,
    )
    # Normalize the generated array through the same canonical image preprocessing path.
    result.image = preprocess_image_only(result.image)
    return {
        'result': result,
        'parameters': {'prompt': prompt, 'width': 1024, 'height': 1024},
    }

generation_case = profile_mode('07-text-to-image', '7. Text-to-Image', generation_workflow)
save_and_display('07-text-to-image', [
    ('output', 'Generated output', generation_case['result'].image),
])

## 8. Outpainting — FLUX Fill
Extend the mountain landscape horizontally. The operation creates its own outpaint mask.

In [ ]:
def outpaint_workflow():
    image = prepare_image('cat.jpg')
    # The 1920 px-wide processed input plus 64 px on each side produces
    # a 2048 px-wide output, exactly matching processing.max_image_side.
    margins = OutpaintMargins(left=128, right=128)
    prompt = 'Continue the original scene naturally on both sides'
    result = processor.outpaint(
        image,
        prompt,
        margins,
        generation_options=GEN_FLUX_FILL,
    )
    return {
        'result': result, 'input': image, 'mask': result.mask,
        'parameters': {
            'input_file': 'cat.jpg',
            'preprocess_max_side': PROCESSING_MAX_SIDE,
            'prompt': prompt,
            'margins': {'left': 128, 'top': 0, 'right': 128, 'bottom': 0},
        },
    }

outpaint_case = profile_mode('08-outpainting', '8. Outpainting', outpaint_workflow)
save_and_display('08-outpainting', [
    ('input', 'Processed input', outpaint_case['input']),
    ('mask', 'Generated outpaint mask', outpaint_case['mask']),
    ('output', 'Output', outpaint_case['result'].image),
])

## 9. Upscaling — Real-ESRGAN
Preprocess the portrait to a 512 px maximum side, then upscale it 2×.

In [ ]:
def upscale_workflow():
    image = prepare_image('demo-portrait.jpg', max_side=512)
    result = processor.upscale(
        image,
        options=UpscaleOptions(scale=2, face_enhance=False),
    )
    return {
        'result': result, 'input': image,
        'parameters': {
            'input_file': 'demo-portrait.jpg',
            'preprocess_max_side': 512,
            'scale': 2,
            'face_enhance': False,
        },
    }

upscale_case = profile_mode('09-upscaling', '9. Upscaling', upscale_workflow)
save_and_display('09-upscaling', [
    ('input', 'Processed input', upscale_case['input']),
    ('output', 'Output', upscale_case['result'].image),
])

## Cleanup
Release all loaded models after the showcase run.

In [ ]:
processor.release_models()
gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except ImportError:
    pass
print('Released all model resources.')